<div style="
  background: linear-gradient(145deg, #0f172a, #1e293b);
  border: 4px solid transparent; border-radius: 14px; padding: 18px 22px; margin: 12px 0;
  font-size: 36px; font-weight: 600; color: #f8fafc;
  box-shadow: 0 6px 14px rgba(0,0,0,0.25); background-clip: padding-box; position: relative;
">
  <div style="position: absolute; inset: 0; padding: 4px; border-radius: 14px;
    background: linear-gradient(90deg, #06b6d4, #3b82f6, #8b5cf6);
    -webkit-mask: linear-gradient(#fff 0 0) content-box, linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor; mask-composite: exclude; pointer-events: none;"></div>
  <b>02. Deep Probabilistic Time Series</b>
</div>

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">📑 Table of Contents</span>

- **1. [Combining Deep Learning with Probabilistic Modeling](#-part-i-deep-probabilistic)**
- **2. [Setup & Imports](#-part-ii-setup--imports)**
- **3. [Probabilistic LSTM with TFP Layers](#-part-iii-probabilistic-lstm)**
    - 3.1 [Architecture Design](#architecture-design)
    - 3.2 [Training with NLL Loss](#training)
- **4. [Uncertainty in Sequential Predictions](#-part-iv-uncertainty)**
- **5. [Key Takeaways](#-part-v-key-takeaways)**

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">1. Combining Deep Learning with Probabilistic Modeling</span>

| **Approach** | **Strengths** | **Limitations** |
|-------------|--------------|----------------|
| STS (tfp.sts) | Interpretable, Bayesian | Limited flexibility |
| LSTM/RNN | Flexible, handles complex patterns | No built-in uncertainty |
| **Probabilistic LSTM** | **Flexible + uncertain** | **Requires more data** |

The key idea: Replace the deterministic output layer of an LSTM with a **probabilistic layer** that outputs a distribution:

$$\text{LSTM}(x_{1:t}) \rightarrow (\mu_t, \sigma_t) \rightarrow y_t \sim \mathcal{N}(\mu_t, \sigma_t^2)$$

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">2. Setup & Imports</span>

In [ ]:
import tensorflow as tf
import tensorflow_probability as tfp
import numpy as np
import matplotlib.pyplot as plt

tfd = tfp.distributions

tf.random.set_seed(42)
np.random.seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

print(f"TensorFlow: {tf.__version__}")
print(f"TFP: {tfp.__version__}")

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">3. Probabilistic LSTM with TFP Layers</span>

In [ ]:
# Generate synthetic sequential data with heteroscedastic noise
np.random.seed(42)
T = 1000
t = np.linspace(0, 20*np.pi, T)

# Signal with time-varying noise
signal = np.sin(t) + 0.5 * np.sin(3*t)
noise_scale = 0.1 + 0.3 * np.abs(np.sin(0.2 * t))  # Heteroscedastic noise
y = (signal + noise_scale * np.random.randn(T)).astype(np.float32)

# Create sequences for LSTM
seq_len = 30
def create_sequences(data, seq_length):
    X, Y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        Y.append(data[i+seq_length])
    return np.array(X, dtype=np.float32), np.array(Y, dtype=np.float32)

X_seq, y_seq = create_sequences(y, seq_len)
X_seq = X_seq[..., np.newaxis]  # Add feature dimension

# Train/test split
split = int(0.8 * len(X_seq))
X_train, X_test = X_seq[:split], X_seq[split:]
y_train_seq, y_test_seq = y_seq[:split], y_seq[split:]

print(f"Training sequences: {X_train.shape}")
print(f"Test sequences: {X_test.shape}")

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">3.1. Architecture Design</span>

In [ ]:
def build_probabilistic_lstm():
    """
    LSTM that outputs a Normal distribution (mean + variance)
    instead of a point prediction.
    """
    model = tf.keras.Sequential([
        tf.keras.layers.LSTM(64, return_sequences=True, input_shape=(seq_len, 1)),
        tf.keras.layers.Dropout(0.1),
        tf.keras.layers.LSTM(32),
        tf.keras.layers.Dense(32, activation='relu'),
        # Output 2 parameters: mean and log_scale
        tf.keras.layers.Dense(2),
        # Convert to a Normal distribution
        tfp.layers.DistributionLambda(
            lambda params: tfd.Normal(
                loc=params[..., 0],
                scale=tf.nn.softplus(params[..., 1]) + 1e-5
            )
        ),
    ])
    return model

prob_lstm = build_probabilistic_lstm()

# Negative log-likelihood loss
nll = lambda y, dist: -dist.log_prob(y)

prob_lstm.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss=nll
)

prob_lstm.summary()

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">3.2. Training</span>

In [ ]:
history = prob_lstm.fit(
    X_train, y_train_seq,
    epochs=50,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(history.history['loss'], label='Train Loss', color='b', linewidth=2)
ax.plot(history.history['val_loss'], label='Val Loss', color='r', linewidth=2)
ax.set_xlabel('Epoch', fontsize=13)
ax.set_ylabel('NLL Loss', fontsize=13)
ax.set_title('Probabilistic LSTM Training', fontsize=15, fontweight='bold')
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">4. Uncertainty in Sequential Predictions</span>

In [ ]:
# Get predictive distributions
pred_dist = prob_lstm(X_test)
pred_mean = pred_dist.mean().numpy()
pred_std = pred_dist.stddev().numpy()

# Plot
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Predictions with uncertainty
test_range = np.arange(len(y_test_seq))
axes[0].plot(test_range, y_test_seq, color='k', linewidth=1.5, alpha=0.7, label='True')
axes[0].plot(test_range, pred_mean, color='b', linewidth=2, label='Predicted mean')
axes[0].fill_between(test_range, pred_mean - 2*pred_std, pred_mean + 2*pred_std,
                     alpha=0.2, color='b', label='±2σ')
axes[0].set_xlabel('Time step', fontsize=13)
axes[0].set_ylabel('Value', fontsize=13)
axes[0].set_title('Probabilistic LSTM: Predictions with Uncertainty', fontsize=15, fontweight='bold')
axes[0].legend(fontsize=11)

# Predicted uncertainty (standard deviation)
axes[1].plot(test_range, pred_std, color='r', linewidth=2)
axes[1].set_xlabel('Time step', fontsize=13)
axes[1].set_ylabel('Predicted σ', fontsize=13)
axes[1].set_title('Predicted Uncertainty (Heteroscedastic)', fontsize=15, fontweight='bold')

plt.tight_layout()
plt.show()

# Calibration
coverage = np.mean(np.abs(y_test_seq - pred_mean) < 2 * pred_std)
print(f"95% CI Coverage: {coverage:.1%} (target: 95%)")

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">5. Key Takeaways</span>

| **Concept** | **Key Point** |
|------------|---------------|
| Probabilistic LSTM | LSTM + `DistributionLambda` → distribution output |
| NLL loss | Train by minimizing negative log-likelihood |
| Heteroscedastic | Model can learn input-dependent noise |
| Calibration | Check that predicted intervals have correct coverage |
| `tfp.layers` | Seamless integration with Keras sequential/functional API |

### 🎯 When to Use Deep Probabilistic Time Series
- Complex, nonlinear temporal patterns
- Need for input-dependent (heteroscedastic) uncertainty
- Large datasets (>1000 time steps)
- Multi-step ahead forecasting with uncertainty